# 06. Residual connection: вход плюс результат слоя

Лекция: [Residual connection](../../notes/interview-prep/08g-residual-connections.md).

Одна новая идея: y = x + F(x). Сегодня достаточно части A — одна строка кода и два вопроса. B/C — готовые небольшие эксперименты, которые разберём вместе после A.

Attention переписывать не нужно. Сначала отделим смысл residual от сложных вычислений внутри F. Это начало Transformer-блока, не полный блок.

## A. Сохранить входной путь и прибавить изменение

Без residual: y = F(x). С residual: y = x + F(x). Сложение покоординатное, не cat. F — преобразование, например attention; в этом упражнении его результат задан явно, чтобы не отвлекаться на внутреннюю формулу.

Слово «поправка» — интерпретация: F(x) не обязано быть маленьким или полезным до обучения.

In [1]:
import torch
from torch import nn

torch.set_printoptions(precision=3, sci_mode=False)
# Один batch, две позиции, по три признака.
x = torch.tensor([[[2., -1., 3.],
                   [0.,  4., 1.]]])
# Условный результат F(x), здесь задан вручную, а не вычислен слоем.
branch_output = torch.tensor([[[0.5, 1., -1.],
                               [2., -0.5, 0.]]])
print('Вход:', x)
print('Результат ветви:', branch_output)

Вход: tensor([[[ 2., -1.,  3.],
         [ 0.,  4.,  1.]]])
Результат ветви: tensor([[[ 0.500,  1.000, -1.000],
         [ 2.000, -0.500,  0.000]]])


In [2]:
y = x + branch_output

assert isinstance(y, torch.Tensor), 'Замени ... сложением'
assert x.shape == branch_output.shape == y.shape == (1, 2, 3)
torch.testing.assert_close(y, torch.tensor([[[2.5, 0., 2.],
                                           [2., 3.5, 1.]]]))
print('После residual:', y)

После residual: tensor([[[2.500, 0.000, 2.000],
         [2.000, 3.500, 1.000]]])


### Остановка A

1. Если F(x) вернёт только нули, чему будет равен y = x + F(x)?
2. Почему здесь используем сложение, а не torch.cat? Какова форма результата?

**Мои ответы:**

1. будет равен входу x
2. Потому что нам нужно добавить влияение контекста, а не сделать его отельным. Форма будет такая же как и входная - B,T,D

Сохрани и пришли на проверку. Не нужно сегодня воспроизводить attention по памяти.

## B. Настоящий слой с нулевыми весами

Готовый опыт: F — Linear(3,3) без bias. Намеренно обнуляем веса, поэтому F(x)=0. Это иллюстрация, не рекомендация инициализировать нулями все слои будущей модели.

No_grad нужен только для ручного изменения весов. Ниже no_grad уже закончился.

In [3]:
branch = nn.Linear(3, 3, bias=False)
with torch.no_grad():
    branch.weight.zero_()

plain_output = branch(x)
residual_output = x + branch(x)
torch.testing.assert_close(plain_output, torch.zeros_like(x))
torch.testing.assert_close(residual_output, x)
print('Без residual:', plain_output)
print('С residual:', residual_output)

# Вход проходит напрямую, но ветвь также может получать градиенты.
branch.zero_grad(set_to_none=True)
probe_loss = residual_output.square().mean()
probe_loss.backward()
assert branch.weight.grad is not None
assert torch.isfinite(branch.weight.grad).all()
assert branch.weight.grad.abs().sum() > 0
print('Градиенты весов ветви:', branch.weight.grad)
# optimizer.step() нет: веса не обновлены.

Без residual: tensor([[[0., 0., 0.],
         [0., 0., 0.]]], grad_fn=<UnsafeViewBackward0>)
С residual: tensor([[[ 2., -1.,  3.],
         [ 0.,  4.,  1.]]], grad_fn=<AddBackward0>)
Градиенты весов ветви: tensor([[ 1.333, -0.667,  2.000],
        [-0.667,  5.667,  0.333],
        [ 2.000,  0.333,  3.333]])


## C. Откуда берётся прямой путь градиента

Упростим всё до одного числа: F(x)=0.1*x. Тогда без residual производная выхода по входу равна 0.1, а с residual — 1.1, потому что y=x+0.1*x.

Сравниваем именно производные выхода по входу: вызываем backward на скалярных выходах. Это не сравнение качества двух моделей и не доказательство, что residual всегда увеличивает градиент.

In [4]:
x_plain = torch.tensor(2.0, requires_grad=True)
x_residual = torch.tensor(2.0, requires_grad=True)

plain = 0.1 * x_plain
with_residual = x_residual + 0.1 * x_residual
plain.backward()
with_residual.backward()

torch.testing.assert_close(x_plain.grad, torch.tensor(0.1))
torch.testing.assert_close(x_residual.grad, torch.tensor(1.1))
print('dy/dx без residual:', x_plain.grad.item())
print('dy/dx с residual:', x_residual.grad.item())

dy/dx без residual: 0.10000000149011612
dy/dx с residual: 1.100000023841858


### После совместного разбора B/C

1. Почему при нулевом F(x) выход residual-слоя сохраняет вход?
2. Почему в опыте B у весов нулевой ветви появились градиенты? Изменились ли сами веса?
3. Откуда взялась единица в производной 1.1 в опыте C?

**Мои ответы:**

1. Потому что на выход мы явно передаем x+F(x). Выход равен входу, если residual равен нулю
2. Потому что у нас сами значения x не нулевые и от них посчитался градиент. Веса не изменились
3. Это из-за residual connection

**Разбор ментора (исходные ответы сохранены):**

Проверки B/C прошли по сохранённым результатам: нулевая ветвь с residual вернула x, веса ветви получили ненулевые градиенты; производные выхода по входу в C равны примерно 0.1 и 1.1.
1. Верно: x+0=x. Под нулём здесь понимается результат ветви F(x), а не отсутствие самого residual-соединения.
2. Веса действительно не изменились: optimizer.step() отсутствует. Объяснение градиента нужно уточнить: производная loss по весу учитывает и чувствительность выхода к весу, и чувствительность loss к выходу. Одного ненулевого x недостаточно для гарантии ненулевого градиента при любом loss. Например, y=x+w*x, loss=y², x=2, w=0: dloss/dw=(2*y)*x=4*2=8. Нулевое значение веса не означает, что его изменение не влияет на loss.
3. Верно по направлению, но назовём источник точно: производная прямого слагаемого x по x равна 1. Для y=x+0.1*x получаем dy/dx=1+0.1=1.1. Здесь, в отличие от B, вычисляем производную выхода по входу, а не loss по весу.

Следующий этап — LayerNorm; residual используем как уже разобранный компонент.

Для нашей модели attention(x) имеет форму (B,T,D), как x, поэтому их можно складывать. В будущем pre-norm блоке будет x + attention(norm(x)); LayerNorm изучим следующим шагом. Residual не гарантирует отсутствие потерь информации или затухания градиентов при любых параметрах.

Перед итоговой проверкой: Restart Kernel → Run All → сохранить outputs.